In [1]:
# ============================================================
# TrustSyn TRUST LAYER v4
# FINAL TRUST SCORE GENERATION
# ============================================================

import pandas as pd
import numpy as np
from pathlib import Path


TRUST_DIR = Path(
    "/Users/konuri/stacking/TrustSyn_TRUST_LAYER"
)


# -----------------------------
# Load trust artifacts
# -----------------------------

ensemble_file = TRUST_DIR / "TrustSyn_ensemble_uncertainty.csv"
conformal_file = TRUST_DIR / "TrustSyn_conformal_predictions.csv"
calibration_file = TRUST_DIR / "TRUST_LAYER_v3_1_CALIBRATION.csv"


ensemble = pd.read_csv(
    ensemble_file
)

conformal = pd.read_csv(
    conformal_file
)

calibration = pd.read_csv(
    calibration_file
)


print("ENSEMBLE:", ensemble.shape)
print("CONFORMAL:", conformal.shape)


# -----------------------------
# Merge
# -----------------------------

df = ensemble.copy()


# add conformal columns
conf_cols = [
    "conformal_lower_90",
    "conformal_upper_90",
    "prediction_interval_width"
]


for c in conf_cols:
    if c in conformal.columns:
        df[c] = conformal[c]


# calibration value
cal_error = float(
    calibration["value"].iloc[0]
)


# -----------------------------
# Trust components
# -----------------------------

# Agreement:
# higher = models agree
agreement = df["agreement_score"]


# uncertainty:
# lower std = better
uncertainty_score = 1 / (
    1 + df["ensemble_std"]
)


# interval confidence:
# narrower interval = better

if "prediction_interval_width" in df.columns:

    interval_score = 1 / (
        1 +
        df["prediction_interval_width"]
    )

else:

    interval_score = np.ones(
        len(df)
    )



# calibration confidence
calibration_score = 1 / (
    1 + cal_error
)


# -----------------------------
# FINAL TRUST SCORE
# -----------------------------

df["trust_score"] = (
    0.4 * agreement +
    0.3 * uncertainty_score +
    0.2 * interval_score +
    0.1 * calibration_score
)


# normalize 0-1

df["trust_score"] = (
    df["trust_score"]
    -
    df["trust_score"].min()
) / (
    df["trust_score"].max()
    -
    df["trust_score"].min()
)



# -----------------------------
# Trust categories
# -----------------------------

df["trust_category"] = pd.cut(
    df["trust_score"],
    bins=[
        -0.01,
        0.33,
        0.66,
        1.0
    ],
    labels=[
        "LOW_TRUST",
        "MEDIUM_TRUST",
        "HIGH_TRUST"
    ]
)


# -----------------------------
# Save
# -----------------------------

out = TRUST_DIR / "TrustSyn_FINAL_TRUST_SCORE.csv"


df.to_csv(
    out,
    index=False
)


print("\n====================")
print("FINAL TRUST TABLE")
print(df.shape)

print(
    df[
        [
            "trust_score",
            "trust_category"
        ]
    ].head()
)


print("\nSAVED:")
print(out)

ENSEMBLE: (88753, 79)
CONFORMAL: (88753, 86)

FINAL TRUST TABLE
(88753, 84)
   trust_score trust_category
0     0.451731   MEDIUM_TRUST
1     0.813202     HIGH_TRUST
2     0.385221   MEDIUM_TRUST
3     0.333934   MEDIUM_TRUST
4     0.978362     HIGH_TRUST

SAVED:
/Users/konuri/stacking/TrustSyn_TRUST_LAYER/TrustSyn_FINAL_TRUST_SCORE.csv
